In [1]:
# ============================================================
# Model 10: Simple Pseudo-Labeling IndoBERT Baseline
# Terminal Table Output Only
# ============================================================

import os
import re
import random
import numpy as np
import pandas as pd
import torch

from collections import Counter
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# ============================================================
# CONFIG
# ============================================================

DATA_DIR = "/net/pr2/projects/plgrid/plggman01/HateSpeech"

PATH_IDHSD = os.path.join(DATA_DIR, "IDHSD_RIO_unbalanced_713_2017.txt")
PATH_572 = os.path.join(DATA_DIR, "572-hate-speech-dataset.csv")
PATH_RE = os.path.join(DATA_DIR, "re_dataset.csv")
PATH_KAMUSALAY = os.path.join(DATA_DIR, "new_kamusalay.csv")

MODEL_NAME = "indobenchmark/indobert-base-p1"
MODEL_LABEL = "SimplePseudoLabel_IndoBERT"

SEED = 42
LABEL_FRACTIONS = [0.05, 0.10, 0.20]

MAX_LEN = 128
BATCH_SIZE = 8
EPOCHS_STAGE1 = 3
EPOCHS_STAGE2 = 5
PSEUDO_THRESHOLD = 0.90

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ============================================================
# SAME DATA PREPROCESSING AS MODELS 6-9
# ============================================================

def safe_read_csv(path, sep=",", header="infer"):
    for enc in ["utf-8", "utf-8-sig", "latin1", "cp1252"]:
        try:
            return pd.read_csv(path, sep=sep, encoding=enc, header=header)
        except Exception:
            pass
    raise ValueError(f"Cannot read {path}")

def normalize_binary_label(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        v = value.strip().lower()
        if v in {"hs", "hate", "hate_speech", "1", "true", "yes"}:
            return 1
        if v in {"non_hs", "non-hs", "non hs", "0", "false", "no"}:
            return 0
    if isinstance(value, (int, float, np.integer, np.floating)):
        return int(value)
    return np.nan

URL_PATTERN = re.compile(r"http\S+|www\S+|https\S+")
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#(\w+)")
NON_ALNUM_PATTERN = re.compile(r"[^a-zA-Z0-9\s!?]")
MULTISPACE_PATTERN = re.compile(r"\s+")
REPEAT_CHAR_PATTERN = re.compile(r"(.)\1{2,}")

LAUGHTER_VARIANTS = {
    "wkwk": "tertawa",
    "wkwkwk": "tertawa",
    "haha": "tertawa",
    "hahaha": "tertawa",
    "hehe": "tertawa"
}

def basic_clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("\\n", " ").replace("\\/", "/")
    text = REPEAT_CHAR_PATTERN.sub(r"\1\1", text)
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = HASHTAG_PATTERN.sub(lambda m: f" {m.group(1)} ", text)
    text = text.lower()
    text = NON_ALNUM_PATTERN.sub(" ", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text

def load_kamusalay_dict(path):
    df = safe_read_csv(path)
    c1, c2 = df.columns[:2]
    slang = {}
    for _, row in df.iterrows():
        src = str(row[c1]).strip().lower()
        tgt = str(row[c2]).strip().lower()
        if src and src != "nan":
            slang[src] = tgt
    return slang

def preprocess_text(text, slang_dict):
    text = basic_clean_text(text)
    tokens = []
    for tok in text.split():
        tok = LAUGHTER_VARIANTS.get(tok, tok)
        tok = slang_dict.get(tok, tok)
        tokens.append(tok)
    return " ".join(tokens)

def load_idhsd_dataset(path):
    df = safe_read_csv(path, sep="\t")
    df = df.rename(columns={"Label":"raw_label","Tweet":"text"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    return df[["text","label"]].dropna()

def load_572_dataset(path):
    df = safe_read_csv(path)
    text_col = next((c for c in ["comment_text","tweet","Tweet","text","Text"] if c in df.columns), None)
    label_col = next((c for c in ["class","label","Label","HS"] if c in df.columns), None)
    df = df.rename(columns={text_col:"text", label_col:"raw_label"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    return df[["text","label"]].dropna()

def load_re_dataset(path):
    df = safe_read_csv(path)
    text_col = next((c for c in ["Tweet","tweet","text","Text"] if c in df.columns), None)
    label_col = "HS" if "HS" in df.columns else "label"
    df = df.rename(columns={text_col:"text", label_col:"raw_label"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    return df[["text","label"]].dropna()

def merge_datasets():
    slang_dict = load_kamusalay_dict(PATH_KAMUSALAY)

    data = pd.concat([
        load_idhsd_dataset(PATH_IDHSD),
        load_572_dataset(PATH_572),
        load_re_dataset(PATH_RE)
    ], ignore_index=True)

    data = data.drop_duplicates(subset=["text"]).reset_index(drop=True)
    data["clean_text"] = data["text"].apply(lambda x: preprocess_text(x, slang_dict))
    data = data[data["clean_text"].str.len() > 2].reset_index(drop=True)

    print("="*70)
    print("FINAL DATASET SIZE :", len(data))
    print("LABEL DISTRIBUTION :", Counter(data["label"]))
    print("MODEL              :", MODEL_LABEL)
    print("="*70)

    return data

# ============================================================
# HF DATASET
# ============================================================

class HFDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# ============================================================
# METRICS
# ============================================================

def evaluate_predictions(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

def find_best_threshold(y_true, y_prob):
    best_t = 0.5
    best_f1 = -1
    for t in np.arange(0.20,0.81,0.02):
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, average="macro")
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    return best_t

def compute_metrics_hf(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:,1]
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

# ============================================================
# TRAINER HELPER
# ============================================================

def train_indobert(train_df, val_df, outdir, epochs):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    train_ds = HFDataset(train_df["clean_text"].tolist(), train_df["label"].astype(int).tolist(), tokenizer)
    val_ds = HFDataset(val_df["clean_text"].tolist(), val_df["label"].astype(int).tolist(), tokenizer)

    args = TrainingArguments(
        output_dir=outdir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        seed=SEED,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics_hf,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()
    return trainer, tokenizer

# ============================================================
# MODEL 10 EXPERIMENT
# ============================================================

def run_experiment(labeled_train_df, unlabeled_df, val_df, test_df, frac):

    # ---------- Stage 1 ----------
    trainer_stage1, tokenizer = train_indobert(
        labeled_train_df,
        val_df,
        f"./tmp_model10_stage1_{frac}",
        EPOCHS_STAGE1
    )

    unlabeled_ds = HFDataset(
        unlabeled_df["clean_text"].tolist(),
        [0]*len(unlabeled_df),
        tokenizer
    )

    logits = trainer_stage1.predict(unlabeled_ds).predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:,1]

    pseudo_mask = (probs >= PSEUDO_THRESHOLD) | (probs <= 1-PSEUDO_THRESHOLD)

    pseudo_texts = unlabeled_df["clean_text"].values[pseudo_mask]
    pseudo_labels = (probs[pseudo_mask] >= PSEUDO_THRESHOLD).astype(int)

    pseudo_df = pd.DataFrame({
        "clean_text": pseudo_texts,
        "label": pseudo_labels
    })

    expanded_train_df = pd.concat([labeled_train_df[["clean_text","label"]], pseudo_df], ignore_index=True)

    # ---------- Stage 2 ----------
    trainer_stage2, tokenizer = train_indobert(
        expanded_train_df,
        val_df,
        f"./tmp_model10_stage2_{frac}",
        EPOCHS_STAGE2
    )

    val_ds = HFDataset(val_df["clean_text"].tolist(), val_df["label"].astype(int).tolist(), tokenizer)
    test_ds = HFDataset(test_df["clean_text"].tolist(), test_df["label"].astype(int).tolist(), tokenizer)

    val_logits = trainer_stage2.predict(val_ds).predictions
    val_prob = torch.softmax(torch.tensor(val_logits), dim=1).numpy()[:,1]
    best_t = find_best_threshold(val_df["label"].astype(int).values, val_prob)

    test_logits = trainer_stage2.predict(test_ds).predictions
    test_prob = torch.softmax(torch.tensor(test_logits), dim=1).numpy()[:,1]
    test_pred = (test_prob >= best_t).astype(int)

    result = evaluate_predictions(test_df["label"].astype(int).values, test_pred, test_prob)
    result["fraction"] = frac
    result["labeled_size"] = len(labeled_train_df)
    result["pseudo_added"] = len(pseudo_df)
    result["threshold"] = best_t

    return result

# ============================================================
# PRINT TABLE
# ============================================================

def print_results_table(results):
    print("\n" + "="*118)
    print("{:<10} {:<12} {:<12} {:<12} {:<10} {:<10} {:<10} {:<10} {:<10}".format(
        "Fraction","Labeled","PseudoAdded","Threshold","Accuracy","MacroF1","Precision","Recall","ROC_AUC"
    ))
    print("="*118)

    for r in results:
        print("{:<10} {:<12} {:<12} {:<12.4f} {:<10.4f} {:<10.4f} {:<10.4f} {:<10.4f} {:<10.4f}".format(
            r["fraction"], r["labeled_size"], r["pseudo_added"], r["threshold"],
            r["accuracy"], r["macro_f1"], r["precision"], r["recall"], r["roc_auc"]
        ))

    print("="*118)

# ============================================================
# MAIN
# ============================================================

data = merge_datasets()

train_val_df, test_df = train_test_split(
    data,
    test_size=0.20,
    random_state=SEED,
    stratify=data["label"]
)

train_df_full, val_df = train_test_split(
    train_val_df,
    test_size=0.10,
    random_state=SEED,
    stratify=train_val_df["label"]
)

results = []

for frac in LABEL_FRACTIONS:
    print("\n" + "="*80)
    print(f"Running Model 10: Simple Pseudo Labeling | Label Fraction = {frac}")
    print("="*80)

    labeled_train_df, unlabeled_df = train_test_split(
        train_df_full,
        train_size=frac,
        random_state=SEED,
        stratify=train_df_full["label"]
    )

    res = run_experiment(labeled_train_df, unlabeled_df, val_df, test_df, frac)
    results.append(res)

print_results_table(results)

print("\nMODEL 10: SIMPLE PSEUDO-LABELING BASELINE FINISHED.")

2026-04-27 00:05:20.276370: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


FINAL DATASET SIZE : 13899
LABEL DISTRIBUTION : Counter({0.0: 7966, 1.0: 5933})
MODEL              : SimplePseudoLabel_IndoBERT

Running Model 10: Simple Pseudo Labeling | Label Fraction = 0.05


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/net/tscratch/people/plgshoffan/shoffan01/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.583467,0.704137,0.662397
2,No log,0.511316,0.758094,0.752093
3,No log,0.491860,0.771583,0.765850


/net/tscratch/people/plgshoffan/shoffan01/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/net/tscratch/people/plgshoffan/shoffan01/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/net/tscratch/people/plgshoffan/shoffan01/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/net/tscratch/people/plgshoffan/shoffan01/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.482848,0.810252,0.809189
2,No log,0.536724,0.818345,0.815983
3,No log,0.633837,0.824640,0.822095
4,No log,0.781448,0.830036,0.827259
5,0.077800,0.793084,0.833633,0.830674


/net/tscratch/people/plgshoffan/shoffan01/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/net/tscratch/people/plgshoffan/shoffan01/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/net/tscratch/people/plgshoffan/shoffan01/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/net/tscratch/people/plgshoffan/shoffan01/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vecto

/net/tscratch/people/plgshoffan/shoffan01/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



Fraction   Labeled      PseudoAdded  Threshold    Accuracy   MacroF1    Precision  Recall     ROC_AUC   
0.05       500          493          0.5600       0.7888     0.7846     0.7842     0.7852     0.8629    
0.1        1000         2870         0.2800       0.8137     0.8088     0.8103     0.8076     0.8917    
0.2        2001         5174         0.2000       0.8475     0.8445     0.8438     0.8454     0.9187    

MODEL 10: SIMPLE PSEUDO-LABELING BASELINE FINISHED.


In [2]:
import os
os.getcwd()

'/net/tscratch/people/plgshoffan/HateSpeech/paper_outputs_hybrid_plus/Experiment Used'